# Linear Regression from Scratch

A compact implementation of ordinary least-squares regression that makes the optimization mechanics visible instead of delegating them to a library estimator.

## Project snapshot

| | |
|---|---|
| **Goal** | Learn linear-regression coefficients with batch gradient descent |
| **Data** | A four-row, two-feature toy dataset used as a deterministic smoke test |
| **Approach** | NumPy implementation with an optional intercept and a scikit-learn-style API |
| **Evaluation** | Training MSE on the worked example; this is an implementation check, not a generalization estimate |
| **Result** | The saved run converges to coefficients `A=0.3512`, `B=0.0476`, intercept `2.3401`, and training MSE `1.0528` |

The emphasis is on translating the loss gradient into readable, reusable code while validating input shapes and fitted state.


## 1. Objective and gradient

For a design matrix $X$, target vector $y$, and coefficient vector $\beta$, linear regression predicts

$$\hat{y} = X\beta.$$

With an intercept, a column of ones is appended to $X$. The implementation minimizes mean squared error:

$$J(\beta)=\frac{1}{n}\lVert y-X\beta\rVert_2^2.$$

Its gradient is

$$\nabla_\beta J=-\frac{2}{n}X^T(y-X\beta),$$

and each batch-gradient step applies $\beta \leftarrow \beta-\eta\nabla_\beta J$, where $\eta$ is the learning rate.


In [1]:
import numpy as np
import pandas as pd

## 2. Estimator implementation

The estimator accepts NumPy arrays or pandas objects, validates dimensionality, exposes learned attributes, and rejects prediction before fitting or with the wrong feature count. Random initialization is reproducible when `random_state` is set.


In [2]:
class LinearRegression:
    def __init__(self, fit_intercept=True, learning_rate=0.05, n_iter=1000, random_state=None):
        self.fit_intercept = fit_intercept
        self.learning_rate = learning_rate
        self.n_iter = n_iter
        self.random_state = random_state

        # sklearn-like learned attributes
        self.coef_ = None
        self.intercept_ = None
        self.n_features_in_ = None
        self.feature_names_in_ = None

    def _to_numpy_X(self, X):
        # Keep sklearn-ish behavior: accept array-like / pandas
        if hasattr(X, "to_numpy"):  # pandas DataFrame
            if hasattr(X, "columns"):
                self.feature_names_in_ = np.array(X.columns, dtype=object)
            X = X.to_numpy()
        else:
            X = np.asarray(X)

        if X.ndim != 2:
            raise ValueError(f"X must be 2D (n_samples, n_features). Got shape {X.shape}.")
        return X.astype(float, copy=False)

    def _to_numpy_y(self, y):
        if hasattr(y, "to_numpy"):  # pandas Series/DataFrame
            y = y.to_numpy()
        else:
            y = np.asarray(y)

        # sklearn accepts (n,) or (n,1) for single target
        if y.ndim == 2 and y.shape[1] == 1:
            y = y.ravel()
        if y.ndim != 1:
            raise ValueError(f"y must be 1D for single-target regression. Got shape {y.shape}.")
        return y.astype(float, copy=False)

    def fit(self, X, y):
        X = self._to_numpy_X(X)
        y = self._to_numpy_y(y)

        n_samples, n_features = X.shape
        self.n_features_in_ = n_features

        rng = np.random.default_rng(self.random_state)

        if self.fit_intercept:
            # Add bias column internally, but expose intercept_ separately (like sklearn)
            X_design = np.c_[X, np.ones(n_samples)]
            beta = rng.standard_normal(n_features + 1)
        else:
            X_design = X
            beta = rng.standard_normal(n_features)

        # Gradient descent on MSE = (1/n) * ||y - Xb||^2
        for _ in range(self.n_iter):
            y_pred = X_design @ beta
            grad = -(2 / n_samples) * (X_design.T @ (y - y_pred))
            beta -= self.learning_rate * grad

        if self.fit_intercept:
            self.coef_ = beta[:n_features]
            self.intercept_ = beta[-1]
        else:
            self.coef_ = beta
            self.intercept_ = 0.0

        return self

    def predict(self, X):
        if self.coef_ is None:
            raise ValueError("This LinearRegression instance is not fitted yet. Call 'fit' first.")

        if hasattr(X, "to_numpy"):
            X = X.to_numpy()
        else:
            X = np.asarray(X)

        if X.ndim != 2:
            raise ValueError(f"X must be 2D (n_samples, n_features). Got shape {X.shape}.")
        if self.n_features_in_ is not None and X.shape[1] != self.n_features_in_:
            raise ValueError(f"X has {X.shape[1]} features, but model was fit with {self.n_features_in_}.")

        X = X.astype(float, copy=False)
        return X @ self.coef_ + self.intercept_


## 3. Worked example

This deliberately small dataset makes it easy to inspect the fitted parameters. Because the same four rows are used for fitting and scoring, the MSE below only confirms that optimization ran as intended; it does not measure out-of-sample performance.


In [3]:
X = pd.DataFrame({"A": [1, 2, 3, 4], "B": [4, 5, 6, 8]})
y = pd.Series([2,5,3,4])

model = LinearRegression(fit_intercept=True, learning_rate=0.01, n_iter=5000, random_state=0)
model.fit(X, y)

predictions = model.predict(X)
training_mse = np.mean((y.to_numpy() - predictions) ** 2)

print("Coefficients:", dict(zip(X.columns, model.coef_.round(4))))
print(f"Intercept: {model.intercept_:.4f}")
print(f"Training MSE: {training_mse:.4f}")


Coefficients: {'A': 0.3512, 'B': 0.0476}
Intercept: 2.3401
Training MSE: 1.0528


## Results and takeaways

The saved example confirms that the vectorized gradient, intercept handling, pandas conversion, and prediction API work together. More importantly, the loss equation maps directly to the two core training lines: compute `grad`, then update `beta`.

This notebook is best read as an algorithmic implementation exercise rather than a predictive modelling case study.


## Limitations and next steps

- The toy example has no held-out set, so the reported MSE does not estimate generalization.
- Gradient descent is sensitive to feature scale and learning rate; add standardization and a convergence tolerance.
- Track the loss history and compare coefficients with the closed-form least-squares solution or scikit-learn.
- Add tests for multi-collinearity, constant features, non-finite values, and convergence failure.
